# 24 — Post processing

Adds raw data of _temperature_monthly.nc, _humidity_monthly.nc, _precipitation_monthly.nc and _population_density.nc

In [3]:
import geopandas as gpd
import httpx
import pandas as pd
import xarray as xr

from common import PROCESSED_DIR, RAW_DIR

variable_raw = RAW_DIR / 'post_processing'
variable_raw.mkdir(parents=True, exist_ok=True)

## 1. Raw scoring inputs

Adds raw data of `_temperature_monthly.nc`, `_humidity_monthly.nc`, `_precipitation_monthly.nc` and `_population_density.nc`.

Layout is wide — one row per `(lat, lon)`, with 12 columns per monthly variable (`temperature_01`..`temperature_12`, `humidity_01`..`humidity_12`, `precipitation_01`..`precipitation_12`) and a scalar `population_density`. `common.load_raw_scoring_inputs` reconstructs the four DataArrays on the shared atlas grid.

In [ ]:
def _monthly_to_wide(da, prefix):
    """Unstack a ``(lat, lon, month)`` DataArray into a ``{prefix}_MM`` wide frame."""
    return (
        da.to_dataframe(name=prefix)
        .unstack('month')[prefix]
        .rename(columns=lambda m: f'{prefix}_{int(m):02d}')
    )

temp_da = xr.open_dataarray(PROCESSED_DIR / '_temperature_monthly.nc')
humidity_da = xr.open_dataarray(PROCESSED_DIR / '_humidity_monthly.nc')
precip_da = xr.open_dataarray(PROCESSED_DIR / '_precipitation_monthly.nc')
density_da = xr.open_dataarray(PROCESSED_DIR / '_population_density.nc')

raw = (
    _monthly_to_wide(temp_da, 'temperature')
    .join(_monthly_to_wide(humidity_da, 'humidity'), how='outer')
    .join(_monthly_to_wide(precip_da, 'precipitation'), how='outer')
    .join(density_da.to_dataframe(name='population_density'), how='outer')
    .reset_index()
)

value_cols = [c for c in raw.columns if c not in ('lat', 'lon')]
raw = raw.dropna(subset=value_cols, how='all').reset_index(drop=True)

out = PROCESSED_DIR / 'raw_scoring_inputs.parquet'
raw.to_parquet(out, index=False)
print(f'wrote {out} ({out.stat().st_size / 1024**2:.1f} MB, {len(raw):,} rows)')
raw.head()